# Pneumonia Detection from Chest X-rays (Deep Learning)

Educational project: binary classification (NORMAL vs PNEUMONIA) of chest X-ray images with a baseline CNN and a MobileNetV2 transfer-learning model.

> **Medical disclaimer:** this is an educational prototype. It is **not** a validated medical device and must not be used to diagnose, treat, or rule out any condition.

**How to run (Google Colab):** Runtime → Change runtime type → GPU, then Runtime → Run all. The dataset is downloaded automatically with `kagglehub`.
Cells are in dependency order (data → models → evaluation → fine-tuning → Grad-CAM → save → predict).

## 1. Environment

In [ ]:
import sys, tensorflow as tf
print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
import random, shutil, json, hashlib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
import tensorflow as tf

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TF", tf.__version__, "| GPU:", bool(tf.config.list_physical_devices("GPU")))

## 2. Dataset

Source: *Chest X-Ray Images (Pneumonia)* on Kaggle (`paultimothymooney/chest-xray-pneumonia`), originally published by Kermany et al. (Cell, 2018). Check the dataset page for its licence and citation requirements before redistributing any images. **Do not commit the dataset to GitHub.**

In [ ]:
import kagglehub
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
print("Downloaded to:", path)
DATA_DIR = Path(path)

In [ ]:
def find_root(base):
    for p in [base, *base.rglob("*")]:
        if "__MACOSX" in str(p): continue
        if p.is_dir() and (p/"train").is_dir() and (p/"test").is_dir():
            return p
    return None

ROOT = find_root(DATA_DIR)
assert ROOT is not None, "train/test folders not found."
print("Dataset root:", ROOT)

EXT = {".jpg", ".jpeg", ".png"}
rows = []
for split in ["train", "val", "test"]:
    for cls in ["NORMAL", "PNEUMONIA"]:
        d = ROOT / split / cls
        n = sum(1 for f in d.iterdir() if f.suffix.lower() in EXT) if d.is_dir() else None
        rows.append((split, cls, n))
counts = pd.DataFrame(rows, columns=["split", "class", "images"])
print(counts.pivot(index="split", columns="class", values="images"))

In [ ]:
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]   # NORMAL=0, PNEUMONIA=1
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
print("Label mapping:", CLASS_TO_IDX)

records, bad = [], []
for split in ["train", "val", "test"]:
    for cls in CLASS_NAMES:
        for f in (ROOT / split / cls).iterdir():
            if f.suffix.lower() not in EXT:
                bad.append((str(f), "unsupported type")); continue
            try:
                with Image.open(f) as im:
                    records.append((str(f), split, cls, im.size[0], im.size[1], im.mode))
            except Exception:
                bad.append((str(f), "unreadable"))

df = pd.DataFrame(records, columns=["path", "split", "class", "w", "h", "mode"])
df["label"] = df["class"].map(CLASS_TO_IDX)
print("Readable images:", len(df), "| Problem files:", len(bad))
print(df["mode"].value_counts())
print(df[["w", "h"]].describe().loc[["min", "mean", "max"]].round(0))

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="split", hue="class", order=["train", "val", "test"])
plt.title("Images per split and class"); plt.show()

In [ ]:
# A few sample images per class (for visual inspection only)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for r, cls in enumerate(CLASS_NAMES):
    paths = df[(df.split == "train") & (df["class"] == cls)].path.sample(4, random_state=SEED).tolist()
    for c in range(4):
        axes[r, c].imshow(Image.open(paths[c]).convert("L"), cmap="gray")
        axes[r, c].set_title(cls); axes[r, c].axis("off")
plt.tight_layout(); plt.show()

## 3. Splits, leakage checks, and input pipeline

* The official `val` folder has only 16 images, so it is **not used**. A validation set is carved out of the official `train` split (80/20, stratified). The official `test` split is kept untouched.
* Exact-duplicate files (MD5) are checked across splits; validation images that are byte-identical to a training image are removed.
* Class weights come from the **training labels only**, to handle the imbalance (about 3:1 pneumonia:normal).

**Known limitation:** the split is random at image level. The dataset contains several images per patient, so images from the same patient can land in both train and validation, which can make validation scores optimistic. A patient-level (grouped) split is listed under future work.

In [ ]:
from sklearn.model_selection import train_test_split

official_train = df[df.split == "train"]
train_df, val_df = train_test_split(official_train, test_size=0.2, stratify=official_train["label"], random_state=SEED)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = df[df.split == "test"].reset_index(drop=True)

def md5(p):
    with open(p, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

for part in (train_df, val_df, test_df):
    part["hash"] = part["path"].map(md5)

P = [set(x["path"]) for x in (train_df, val_df, test_df)]
assert not (P[0] & P[1]) and not (P[0] & P[2]) and not (P[1] & P[2]), "Path leakage!"
H = [set(x["hash"]) for x in (train_df, val_df, test_df)]
print("identical files  train-val:", len(H[0] & H[1]), "| train-test:", len(H[0] & H[2]), "| val-test:", len(H[1] & H[2]))
print("duplicates inside train:", train_df["hash"].duplicated().sum())
val_df = val_df[~val_df["hash"].isin(H[0])].reset_index(drop=True)   # drop val images that also exist in train

IMG_SIZE = (224, 224)
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_img(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img.set_shape(IMG_SIZE + (3,))
    return tf.cast(img, tf.float32), label

def make_ds(frame, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((frame["path"].tolist(), frame["label"].values.astype("float32").reshape(-1, 1)))
    if shuffle:
        ds = ds.shuffle(len(frame), seed=SEED)
    return ds.map(load_img, num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df), make_ds(test_df)
x, y = next(iter(train_ds))
assert x.shape[1:] == (224, 224, 3) and y.shape[1:] == (1,)
print("Batch:", x.shape, y.shape, "| pixels:", float(tf.reduce_min(x)), "to", float(tf.reduce_max(x)))

n, n0, n1 = len(train_df), int((train_df.label == 0).sum()), int((train_df.label == 1).sum())
class_weight = {0: n / (2 * n0), 1: n / (2 * n1)}
print("train:", len(train_df), "| val:", len(val_df), "| test:", len(test_df))
print("class_weight:", {k: round(v, 3) for k, v in class_weight.items()})

## 4. Models and training

* **Model A – baseline CNN:** 4 conv blocks (16-32-64-128 filters), global average pooling, dropout 0.4, sigmoid output.
* **Model B – MobileNetV2:** ImageNet-pretrained backbone, frozen, with a small classification head.
* Augmentation (small rotation/zoom/translation/contrast) runs inside the model and is only active during training. Pixel scaling is also inside the model, so inputs are raw 0–255 RGB.
* Early stopping and checkpointing monitor **validation AUC**.

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

MODEL_DIR = Path("/content/models") if Path("/content").exists() else Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

augment = tf.keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomContrast(0.1),
], name="augment")

def make_metrics():
    return [tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.AUC(name="auc")]

def make_callbacks(name):
    return [
        EarlyStopping(monitor="val_auc", mode="max", patience=4, restore_best_weights=True),
        ModelCheckpoint(str(MODEL_DIR / f"{name}_best.keras"), monitor="val_auc", mode="max", save_best_only=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
    ]

# ---- Model A: baseline CNN ----
def build_cnn():
    inp = layers.Input((224, 224, 3))
    x = augment(inp)
    x = layers.Rescaling(1.0 / 255)(x)
    for f in [16, 32, 64, 128]:
        x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    return models.Model(inp, out, name="baseline_cnn")

cnn = build_cnn()
cnn.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=make_metrics())
hist_cnn = cnn.fit(train_ds, validation_data=val_ds, epochs=12, class_weight=class_weight, callbacks=make_callbacks("baseline_cnn"))

In [ ]:
# ---- Model B: MobileNetV2 (frozen backbone) ----
def build_mobilenet():
    base = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights="imagenet")
    base.trainable = False
    inp = layers.Input((224, 224, 3))
    x = augment(inp)
    x = layers.Rescaling(1.0 / 127.5, offset=-1)(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    return models.Model(inp, out, name="mobilenetv2"), base

mnet, base = build_mobilenet()
mnet.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=make_metrics())
hist_mnet = mnet.fit(train_ds, validation_data=val_ds, epochs=10, class_weight=class_weight, callbacks=make_callbacks("mobilenetv2"))

In [ ]:
# ---- training curves ----
fig, ax = plt.subplots(2, 2, figsize=(11, 7))
for col, (name, h) in enumerate([("Baseline CNN", hist_cnn), ("MobileNetV2", hist_mnet)]):
    ax[0, col].plot(h.history["accuracy"], label="train")
    ax[0, col].plot(h.history["val_accuracy"], label="val")
    ax[0, col].set_title(name + " accuracy"); ax[0, col].legend()
    ax[1, col].plot(h.history["loss"], label="train")
    ax[1, col].plot(h.history["val_loss"], label="val")
    ax[1, col].set_title(name + " loss"); ax[1, col].legend()
plt.tight_layout(); plt.show()

## 5. Evaluation on the held-out test set

Metrics: accuracy, precision, recall (sensitivity), specificity, F1, ROC-AUC, and the confusion matrix. The model for the next stage is chosen using **validation AUC only**, never test scores.

In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report, roc_auc_score,
                             roc_curve, accuracy_score, precision_score, recall_score, f1_score)

y_true = test_df["label"].values
THRESH = 0.5   # default cut-off, not automatically the best one medically

def evaluate(model, name):
    scores = model.predict(test_ds, verbose=0).ravel()
    pred = (scores >= THRESH).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {"model": name, "accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred),
            "recall (sensitivity)": recall_score(y_true, pred),
            "specificity": tn / (tn + fp),
            "F1": f1_score(y_true, pred),
            "ROC-AUC": roc_auc_score(y_true, scores),
            "TN": tn, "FP": fp, "FN": fn, "TP": tp}, scores, pred

res_cnn, s_cnn, p_cnn = evaluate(cnn, "Baseline CNN")
res_mnet, s_mnet, p_mnet = evaluate(mnet, "MobileNetV2")
results = pd.DataFrame([res_cnn, res_mnet]).set_index("model")
print(results.round(3).T)

val_cnn = max(hist_cnn.history["val_auc"])
val_mnet = max(hist_mnet.history["val_auc"])
print("\nBest validation AUC -> CNN:", round(val_cnn, 3), "| MobileNetV2:", round(val_mnet, 3))
if val_mnet >= val_cnn:
    best_name, best_model, best_scores, best_pred = "MobileNetV2", mnet, s_mnet, p_mnet
else:
    best_name, best_model, best_scores, best_pred = "Baseline CNN", cnn, s_cnn, p_cnn
print("Selected model (by validation):", best_name)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, (name, pred) in zip(ax[:2], [("Baseline CNN", p_cnn), ("MobileNetV2", p_mnet)]):
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=a, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    a.set_title(name + " (test)"); a.set_xlabel("Predicted"); a.set_ylabel("True")
for name, s in [("Baseline CNN", s_cnn), ("MobileNetV2", s_mnet)]:
    fpr, tpr, _ = roc_curve(y_true, s)
    ax[2].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_true, s):.3f})")
ax[2].plot([0, 1], [0, 1], "k--")
ax[2].set_title("ROC curve (test)"); ax[2].set_xlabel("False positive rate"); ax[2].set_ylabel("True positive rate"); ax[2].legend()
plt.tight_layout(); plt.show()

print("Classification report for", best_name, "(threshold 0.5):")
print(classification_report(y_true, best_pred, target_names=CLASS_NAMES, digits=3))

In [ ]:
# Bootstrap 95% confidence intervals on the test set (for the selected model at this stage)
rng = np.random.default_rng(SEED)
accs, aucs = [], []
for _ in range(1000):
    idx = rng.integers(0, len(y_true), len(y_true))
    if len(set(y_true[idx])) < 2:
        continue
    accs.append(accuracy_score(y_true[idx], best_pred[idx]))
    aucs.append(roc_auc_score(y_true[idx], best_scores[idx]))
print(best_name, "accuracy 95% CI:", np.percentile(accs, [2.5, 97.5]).round(3))
print(best_name, "ROC-AUC 95% CI:", np.percentile(aucs, [2.5, 97.5]).round(3))

In [ ]:
# Error analysis: a few misclassified and correctly classified test images
wrong = np.where(best_pred != y_true)[0]
right = np.where(best_pred == y_true)[0]
print("Misclassified:", len(wrong), "of", len(y_true))
rng = np.random.default_rng(SEED)
show = list(rng.choice(wrong, min(4, len(wrong)), replace=False)) + list(rng.choice(right, 4, replace=False))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for a, i in zip(axes.ravel(), show):
    a.imshow(Image.open(test_df.path[i]).convert("L"), cmap="gray")
    ok = best_pred[i] == y_true[i]
    a.set_title(f"True: {CLASS_NAMES[y_true[i]]}\nPred: {CLASS_NAMES[best_pred[i]]} (score {best_scores[i]:.2f})\n{'CORRECT' if ok else 'WRONG'}",
                fontsize=9, color="green" if ok else "red")
    a.axis("off")
plt.tight_layout(); plt.show()

## 6. Decision threshold (chosen on validation data)

For screening-style tasks, missing a pneumonia case (false negative) is usually worse than a false alarm, so the threshold is worth tuning. It is picked on the **validation** set (Youden's J) and then applied once to the test set. This cell must run **before** fine-tuning, because fine-tuning updates `mnet` in place.

In [ ]:
val_scores = mnet.predict(val_ds, verbose=0).ravel()
y_val = val_df["label"].values
fpr, tpr, thr = roc_curve(y_val, val_scores)
best_t = float(thr[np.argmax(tpr - fpr)])   # balances sensitivity and specificity on validation
print("Chosen threshold (from validation):", round(best_t, 3))

test_scores = mnet.predict(test_ds, verbose=0).ravel()
pred_t = (test_scores >= best_t).astype(int)
tn, fp, fn, tp = confusion_matrix(y_true, pred_t, labels=[0, 1]).ravel()
print("Test with tuned threshold -> accuracy:", round(accuracy_score(y_true, pred_t), 3),
      "| recall:", round(tp / (tp + fn), 3), "| specificity:", round(tn / (tn + fp), 3))
print("TN", tn, "FP", fp, "FN", fn, "TP", tp)

## 7. Fine-tuning MobileNetV2

Unfreeze the top 30 backbone layers (BatchNorm layers stay frozen) and train with a small learning rate (1e-5).

In [ ]:
mnet.save_weights(MODEL_DIR / "mnet_before_finetune.weights.h5")   # backup of the frozen-stage model

base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False          # only the top 30 layers can change
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False      # keep BatchNorm stable

mnet.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="binary_crossentropy", metrics=make_metrics())
hist_ft = mnet.fit(train_ds, validation_data=val_ds, epochs=8, class_weight=class_weight, callbacks=make_callbacks("mobilenetv2_finetuned"))

In [ ]:
res_ft, s_ft, p_ft = evaluate(mnet, "MobileNetV2 fine-tuned")
val_ft = max(hist_ft.history["val_auc"])
print("Best validation AUC -> frozen:", round(val_mnet, 3), "| fine-tuned:", round(val_ft, 3))
compare = pd.DataFrame([res_cnn, res_mnet, res_ft]).set_index("model")
print(compare.round(3).T)

## 8. Grad-CAM (qualitative explanation)

Heat-maps show which image regions most influenced the fine-tuned model's output. They are a debugging aid, **not** evidence that the model is looking at clinically meaningful features.

In [ ]:
conv_layer = base.get_layer("out_relu")
grad_model = tf.keras.Model([base.input], [conv_layer.output, base.output])

W, b = mnet.layers[-1].get_weights()   # final Dense layer
W = tf.constant(W); b = tf.constant(b)

def gradcam(arr):
    x = tf.convert_to_tensor(arr[None], dtype=tf.float32)
    x = x / 127.5 - 1.0     # MobileNetV2 preprocessing
    with tf.GradientTape() as tape:
        conv_outputs, _ = grad_model(x, training=False)
        tape.watch(conv_outputs)
        pooled = tf.reduce_mean(conv_outputs, axis=[1, 2])
        logit = tf.matmul(pooled, W) + b
    grads = tape.gradient(logit, conv_outputs)
    assert grads is not None, "No gradient computed!"
    weights = tf.reduce_mean(grads, axis=(1, 2))
    cam = tf.nn.relu(tf.reduce_sum(conv_outputs * weights[:, None, None, :], axis=-1))[0].numpy()
    cam = cam / (cam.max() + 1e-8)
    return tf.image.resize(cam[..., None], IMG_SIZE).numpy()[..., 0]

pred_m = (s_ft >= THRESH).astype(int)
tp_idx = np.where((pred_m == 1) & (y_true == 1))[0]
fp_idx = np.where((pred_m == 1) & (y_true == 0))[0]
rng = np.random.default_rng(SEED)
picks = list(rng.choice(tp_idx, 2, replace=False)) + list(rng.choice(fp_idx, 2, replace=False))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for k, i in enumerate(picks):
    arr = tf.image.resize(np.array(Image.open(test_df.path[i]).convert("RGB"), dtype="float32"), IMG_SIZE).numpy()
    cam = gradcam(arr)
    axes[0, k].imshow(arr.astype("uint8"))
    axes[0, k].set_title("True: " + CLASS_NAMES[y_true[i]] + " | Pred: PNEUMONIA", fontsize=9)
    axes[1, k].imshow(arr.astype("uint8")); axes[1, k].imshow(cam, cmap="jet", alpha=0.4)
    axes[1, k].set_title("Grad-CAM overlay", fontsize=9)
    axes[0, k].axis("off"); axes[1, k].axis("off")
plt.tight_layout(); plt.show()

## 9. Save the final model

The final model is the **fine-tuned MobileNetV2**, because it has the highest validation AUC. (The original notebook saved the pre-fine-tuning model by mistake; this version saves the fine-tuned one.) The weights file is **not** committed to GitHub; see `models/README.md`.

In [ ]:
final_name, final_model = "MobileNetV2 (fine-tuned)", mnet
print("Validation AUC -> frozen:", round(val_mnet, 3), "| fine-tuned:", round(val_ft, 3))

MODEL_PATH = MODEL_DIR / "pneumonia_classifier.keras"
final_model.save(MODEL_PATH)

config = {
    "selected_model": final_name,
    "class_mapping": {"NORMAL": 0, "PNEUMONIA": 1},
    "img_size": [224, 224],
    "threshold": THRESH,
    "preprocessing": "RGB, resize to 224x224, raw pixel values 0-255 (scaling is inside the model)",
    "disclaimer": "Educational prototype. Not a medical device.",
}
with open(MODEL_DIR / "model_config.json", "w") as f:
    json.dump(config, f, indent=2)

loaded = tf.keras.models.load_model(MODEL_PATH)
a = final_model.predict(test_ds.take(2), verbose=0)
b = loaded.predict(test_ds.take(2), verbose=0)
print("Saved and reloaded. Max difference in predictions:", float(np.abs(a - b).max()))

## 10. Predict on your own X-ray

Upload a chest X-ray image (JPG/PNG). Output is a demo score only.

In [ ]:
def predict_image(path, model, threshold=THRESH):
    try:
        Image.open(path).verify()
        img = Image.open(path).convert("RGB")
    except Exception as e:
        print("Invalid image file:", e)
        return
    arr = tf.image.resize(np.array(img, dtype="float32"), IMG_SIZE)
    score = float(model.predict(arr[None], verbose=0)[0, 0])
    label = CLASS_NAMES[int(score >= threshold)]
    plt.figure(figsize=(4, 4))
    plt.imshow(img.convert("L"), cmap="gray"); plt.axis("off")
    plt.title(f"Model output: {label} (score {score:.2f})")
    plt.show()
    print("EDUCATIONAL DEMO ONLY. This cannot confirm or rule out pneumonia in a real person.")

try:
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        predict_image(name, loaded)
except ImportError:
    # Not in Colab: set the path to your own image
    # predict_image("my_xray.jpg", loaded)
    print("Not running in Colab. Call predict_image('path/to/image.jpg', loaded) instead.")

## 11. Download the trained files (Colab)

In [ ]:
try:
    from google.colab import files
    shutil.make_archive("/content/pneumonia_project", "zip", MODEL_DIR)
    files.download("/content/pneumonia_project.zip")
except ImportError:
    print("Not in Colab; model files are in:", MODEL_DIR)